# Load the dataset
Start any new notebook from this one. Always take `class_id` from `classes.csv`: `val/` and `test/` lack some species folders, so loaders that number classes by folder (plain `ImageFolder`) would mislabel photos.

In [ ]:
from pathlib import Path

import kagglehub
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

root = Path(kagglehub.dataset_download("puneetbajajiitm/serpentguard-indian-snakes-clean"))
meta = pd.read_csv(root / "metadata.csv", dtype={"obs_id": str, "photo_id": str, "observer_id": str}, low_memory=False)
classes = pd.read_csv(root / "classes.csv")
CLASS_ID = dict(zip(classes["species"], classes["class_id"]))


class SnakeSplit(Dataset):
    def __init__(self, split, transform=None):
        rows = meta[meta["split"] == split]
        self.paths = [root / "data" / path for path in rows["path"]]
        self.labels = [CLASS_ID.get(species, -1) for species in rows["species"]]  # -1: unknown_pool
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        return (self.transform(image) if self.transform else image), self.labels[index]


for split in ("train", "val", "test", "test_extra", "unknown_pool"):
    print(f"{split:<13} {len(SnakeSplit(split)):>7,} photos")
classes.head(10)